# Malaria Classification — Clean Reproducibility & Integrity Audit

**Purpose:** independently reproduce and audit the reported experimental results **without retraining**.

This notebook verifies:

1. environment provenance;
2. all 5 frozen group-aware splits;
3. train/validation/test leakage;
4. all **30** prediction archives;
5. recomputed Accuracy, Precision, Recall, F1, ROC-AUC, Brier, and confusion matrices;
6. agreement with `results/all_results.csv`;
7. repeated-split statistics;
8. validation-selected threshold analysis;
9. Brier / simulated-prevalence analysis;
10. binary structural integrity of all **25 CNN Keras models**;
11. optional full Keras deserialization.

**Design actually audited**

- Main experiment: 4 configurations × 5 data splits = **20 CNN runs**
- Color-only baseline: **5 runs**
- Blur baseline: **5 CNN runs**
- Total: **30 runs**

All new outputs are written to:

`malaria_colorspace/reproducibility_audit/`

Original experimental artifacts are never overwritten.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, sys, json, glob, hashlib, zipfile, warnings, platform, time
import numpy as np
import pandas as pd

PROJECT_NAME = "malaria_colorspace"
DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJ_DIR = DRIVE_ROOT / PROJECT_NAME

ART_DIR = PROJ_DIR / "artifacts"
SPLIT_DIR = PROJ_DIR / "splits"
RESULTS_DIR = PROJ_DIR / "results"
FIGURES_DIR = PROJ_DIR / "figures"
LEDGER_PATH = PROJ_DIR / "run_ledger.json"
RESULTS_CSV = RESULTS_DIR / "all_results.csv"

AUDIT_DIR = PROJ_DIR / "reproducibility_audit"
AUDIT_FIG_DIR = AUDIT_DIR / "figures"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_FIG_DIR.mkdir(parents=True, exist_ok=True)

required_dirs = [PROJ_DIR, ART_DIR, SPLIT_DIR, RESULTS_DIR]
missing_dirs = [str(p) for p in required_dirs if not p.exists()]
assert not missing_dirs, f"Missing required directories: {missing_dirs}"
assert RESULTS_CSV.exists(), f"Missing: {RESULTS_CSV}"

print("Project:", PROJ_DIR)
print("Audit output:", AUDIT_DIR)

## 1. Environment provenance

Verified original runtime evidence:

- Python 3.13.15
- TensorFlow 2.20.0
- Keras save metadata 3.13.2
- NumPy 2.1.3
- pandas 2.2.3
- scikit-learn 1.6.1
- SciPy 1.16.3
- Matplotlib 3.10.0
- NVIDIA T4

Exact CUDA, cuDNN, and NVIDIA driver versions were not preserved and therefore are not guessed here.

In [ ]:
from importlib import metadata

EXPECTED = {
    "python": "3.13.15",
    "tensorflow": "2.20.0",
    "keras_saved_model": "3.13.2",
    "numpy": "2.1.3",
    "pandas": "2.2.3",
    "scikit-learn": "1.6.1",
    "scipy": "1.16.3",
    "matplotlib": "3.10.0",
    "gpu": "NVIDIA T4",
}

def package_version(name):
    try:
        return metadata.version(name)
    except Exception:
        return None

current_env = {
    "python": platform.python_version(),
    "tensorflow": package_version("tensorflow"),
    "keras": package_version("keras"),
    "numpy": package_version("numpy"),
    "pandas": package_version("pandas"),
    "scikit-learn": package_version("scikit-learn"),
    "scipy": package_version("scipy"),
    "matplotlib": package_version("matplotlib"),
}

try:
    import tensorflow as tf
    current_env["gpu_detected"] = [str(x) for x in tf.config.list_physical_devices("GPU")]
except Exception as e:
    current_env["gpu_detected"] = f"TensorFlow unavailable: {e}"

env_rows = []
for component, verified in EXPECTED.items():
    env_rows.append({
        "component": component,
        "verified_original": verified,
        "current_runtime": current_env.get(component),
        "evidence_class": (
            "verified from .keras metadata"
            if component == "keras_saved_model"
            else "verified original runtime"
        ),
    })

env_df = pd.DataFrame(env_rows)
display(env_df)
env_df.to_csv(AUDIT_DIR / "environment_comparison.csv", index=False)
print("Current GPU:", current_env.get("gpu_detected"))

## 2. Frozen split leakage audit

A split passes only when there is no overlap between train, validation, and test at both:

- inferred `group` level;
- exact `filepath` level.

In [ ]:
SPLIT_SEEDS = [42, 123, 7, 2024, 99]
split_files = sorted(SPLIT_DIR.glob("split_*.csv"))
assert len(split_files) == 5, f"Expected 5 split files, found {len(split_files)}"

required_split_cols = {"filepath", "label", "group", "subset"}
split_rows = []
split_frames = {}
fingerprints = {}

def sha256_strings(values):
    return hashlib.sha256("\n".join(values).encode("utf-8")).hexdigest()

for split_i, f in enumerate(split_files):
    s = pd.read_csv(f)
    split_frames[split_i] = s

    assert required_split_cols.issubset(s.columns)
    assert set(s.subset.unique()) == {"train", "val", "test"}

    tr = s[s.subset == "train"]
    va = s[s.subset == "val"]
    te = s[s.subset == "test"]

    tg, vg, eg = set(tr.group), set(va.group), set(te.group)
    tp, vp, ep = set(tr.filepath), set(va.filepath), set(te.filepath)

    g_tv = len(tg & vg)
    g_tt = len(tg & eg)
    g_vt = len(vg & eg)

    p_tv = len(tp & vp)
    p_tt = len(tp & ep)
    p_vt = len(vp & ep)

    missing = int(s[["filepath", "label", "group", "subset"]].isna().sum().sum())
    dup_paths = int(s.filepath.duplicated().sum())

    fp_pairs = (
        s[["filepath", "label"]]
        .sort_values(["filepath", "label"])
        .astype(str).agg("|".join, axis=1).tolist()
    )
    dataset_hash = sha256_strings(fp_pairs)
    fingerprints[split_i] = dataset_hash

    passed = all(x == 0 for x in [
        g_tv, g_tt, g_vt, p_tv, p_tt, p_vt, missing, dup_paths
    ])

    split_rows.append({
        "split": split_i,
        "seed": SPLIT_SEEDS[split_i],
        "file": f.name,
        "n_total": len(s),
        "n_train": len(tr),
        "n_val": len(va),
        "n_test": len(te),
        "train_groups": tr.group.nunique(),
        "val_groups": va.group.nunique(),
        "test_groups": te.group.nunique(),
        "train_positive": int((tr.label == 1).sum()),
        "val_positive": int((va.label == 1).sum()),
        "test_positive": int((te.label == 1).sum()),
        "train_prevalence": float(tr.label.mean()),
        "val_prevalence": float(va.label.mean()),
        "test_prevalence": float(te.label.mean()),
        "group_train_val_overlap": g_tv,
        "group_train_test_overlap": g_tt,
        "group_val_test_overlap": g_vt,
        "path_train_val_overlap": p_tv,
        "path_train_test_overlap": p_tt,
        "path_val_test_overlap": p_vt,
        "duplicate_paths": dup_paths,
        "missing_values": missing,
        "dataset_sha256": dataset_hash,
        "status": "PASS" if passed else "FAIL",
    })

split_audit = pd.DataFrame(split_rows)
display(split_audit)
split_audit.to_csv(AUDIT_DIR / "split_leakage_audit_recomputed.csv", index=False)

assert (split_audit.status == "PASS").all()
assert len(set(fingerprints.values())) == 1
print("PASS: 5/5 split files have zero group leakage and zero filepath leakage.")
print("Dataset fingerprint:", next(iter(fingerprints.values())))

In [ ]:
import re

s0 = split_frames[0]
groups = pd.Series(sorted(s0.group.astype(str).unique()), name="group")

def group_type(g):
    if re.fullmatch(r"C\d+P\d+", g):
        return "patient-like_CxPy"
    if re.fullmatch(r"IMG_\d+_\d+", g):
        return "fallback_IMG"
    return "other"

group_summary = groups.map(group_type).value_counts().rename_axis("group_type").reset_index(name="n_groups")
display(group_summary)
group_summary.to_csv(AUDIT_DIR / "group_identifier_summary.csv", index=False)
print("Unique groups:", s0.group.nunique())

## 3. Audit all 30 `preds.npz` archives

CNN runs are expected to contain `y_val`, `p_val`, `y_test`, `p_test`.

Color-only baseline runs are expected to contain `y_test`, `p_test`.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, brier_score_loss
)

original_results = pd.read_csv(RESULTS_CSV)
assert len(original_results) == 30
assert original_results.run_id.is_unique

METRICS = ["accuracy", "precision", "recall", "f1", "auc", "brier"]
CONF = ["tn", "fp", "fn", "tp"]

def recompute_metrics(y, p, threshold=0.5):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    yhat = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0,1]).ravel()
    return {
        "accuracy": accuracy_score(y, yhat),
        "precision": precision_score(y, yhat, zero_division=0),
        "recall": recall_score(y, yhat, zero_division=0),
        "f1": f1_score(y, yhat, zero_division=0),
        "auc": roc_auc_score(y, p),
        "brier": brier_score_loss(y, p),
        "n_test": len(y),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

audit_rows = []

for _, ref in original_results.sort_values(["tag", "split", "run_id"]).iterrows():
    run_id = ref.run_id
    pred_path = ART_DIR / run_id / "preds.npz"

    row = {"run_id": run_id, "tag": ref.tag, "split": int(ref.split)}

    if not pred_path.exists():
        row.update(status="FAIL", reason="missing preds.npz")
        audit_rows.append(row)
        continue

    try:
        with np.load(pred_path, allow_pickle=False) as d:
            keys = set(d.files)
            required = {"y_test", "p_test"}
            if ref.tag != "baseline_color":
                required |= {"y_val", "p_val"}

            missing = sorted(required - keys)
            if missing:
                row.update(status="FAIL", reason=f"missing arrays: {missing}")
                audit_rows.append(row)
                continue

            y = np.asarray(d["y_test"])
            p = np.asarray(d["p_test"])

        structural_ok = (
            len(y) == len(p) and len(y) > 0
            and np.isfinite(y).all()
            and np.isfinite(p).all()
            and ((p >= 0) & (p <= 1)).all()
            and set(np.unique(y)).issubset({0,1})
        )

        row.update({
            "npz_keys": ",".join(sorted(keys)),
            "n_test_binary": len(y),
            "finite_probabilities": bool(np.isfinite(p).all()),
            "probability_range_ok": bool(((p >= 0) & (p <= 1)).all()),
            "structural_ok": structural_ok,
            "pred_sha256": hashlib.sha256(pred_path.read_bytes()).hexdigest(),
        })

        if not structural_ok:
            row.update(status="FAIL", reason="binary structural validation failed")
            audit_rows.append(row)
            continue

        rec = recompute_metrics(y, p)
        max_diff = 0.0

        for m in METRICS:
            diff = abs(float(rec[m]) - float(ref[m]))
            row[f"recomputed_{m}"] = rec[m]
            row[f"reported_{m}"] = float(ref[m])
            row[f"absdiff_{m}"] = diff
            max_diff = max(max_diff, diff)

        for c in CONF:
            row[f"recomputed_{c}"] = rec[c]
            row[f"reported_{c}"] = int(ref[c])

        exact_conf = all(int(rec[c]) == int(ref[c]) for c in CONF)
        n_match = int(rec["n_test"]) == int(ref["n_test"])

        row.update({
            "n_test_match": n_match,
            "confusion_exact_match": exact_conf,
            "max_metric_absdiff": max_diff,
            "status": "PASS" if (max_diff <= 1e-6 and exact_conf and n_match) else "WARNING",
            "reason": "",
        })

    except Exception as e:
        row.update(status="FAIL", reason=repr(e))

    audit_rows.append(row)

prediction_audit = pd.DataFrame(audit_rows)
display(prediction_audit[[
    "run_id", "tag", "split", "status",
    "n_test_binary", "max_metric_absdiff", "confusion_exact_match"
]])

prediction_audit.to_csv(AUDIT_DIR / "prediction_integrity_audit_30runs.csv", index=False)

print(prediction_audit.status.value_counts())
assert (prediction_audit.status == "PASS").all()
print("PASS: 30/30 prediction archives reproduce reported metrics.")
print("Maximum absolute metric difference:", prediction_audit.max_metric_absdiff.max())

In [ ]:
consistency_rows = []

for split_i in range(5):
    sub = original_results[original_results.split == split_i].sort_values("run_id")
    y_hashes, p_hashes = {}, {}

    for run_id in sub.run_id:
        with np.load(ART_DIR / run_id / "preds.npz", allow_pickle=False) as d:
            y = np.asarray(d["y_test"])
            p = np.asarray(d["p_test"])
        y_hashes[run_id] = hashlib.sha256(y.tobytes()).hexdigest()
        p_hashes[run_id] = hashlib.sha256(p.tobytes()).hexdigest()

    consistency_rows.append({
        "split": split_i,
        "n_runs": len(sub),
        "unique_y_test_hashes": len(set(y_hashes.values())),
        "unique_p_test_hashes": len(set(p_hashes.values())),
        "same_test_labels_for_all_methods": len(set(y_hashes.values())) == 1,
        "all_probability_outputs_distinct": len(set(p_hashes.values())) == len(sub),
    })

consistency_df = pd.DataFrame(consistency_rows)
display(consistency_df)
consistency_df.to_csv(AUDIT_DIR / "within_split_prediction_consistency.csv", index=False)

assert consistency_df.same_test_labels_for_all_methods.all()
assert consistency_df.all_probability_outputs_distinct.all()
print("PASS: y_test is identical across methods within each split and probability outputs are distinct.")

## 4. Main experiment summary and paired RGB-vs-grayscale statistics

The inferential test below reproduces the original notebook calculation.  
Interpret p-values cautiously because the five repeated partitions come from one dataset.

In [ ]:
exp1 = original_results[original_results.tag == "exp1"].copy()

main_summary = (
    exp1.groupby(["scheme", "finetune"])
    .agg(
        n=("run_id", "count"),
        accuracy_mean=("accuracy", "mean"),
        accuracy_sd=("accuracy", "std"),
        precision_mean=("precision", "mean"),
        precision_sd=("precision", "std"),
        recall_mean=("recall", "mean"),
        recall_sd=("recall", "std"),
        f1_mean=("f1", "mean"),
        f1_sd=("f1", "std"),
        auc_mean=("auc", "mean"),
        auc_sd=("auc", "std"),
        brier_mean=("brier", "mean"),
        brier_sd=("brier", "std"),
    ).reset_index()
)

display(main_summary)
main_summary.to_csv(AUDIT_DIR / "exp1_summary_recomputed.csv", index=False)

In [ ]:
from scipy import stats

def paired_rgb_gray(metric, finetune):
    a = exp1[(exp1.scheme=="rgb") & (exp1.finetune==finetune)].sort_values("split")[metric].to_numpy()
    b = exp1[(exp1.scheme=="grayscale") & (exp1.finetune==finetune)].sort_values("split")[metric].to_numpy()
    assert len(a) == len(b) == 5
    diff = a - b
    t, p = stats.ttest_rel(a, b)
    ci = stats.t.interval(0.95, len(diff)-1, loc=diff.mean(), scale=stats.sem(diff))
    d = diff.mean() / (diff.std(ddof=1) + 1e-9)
    return {
        "metric": metric,
        "finetune": finetune,
        "mean_diff_rgb_minus_gray": round(float(diff.mean()), 4),
        "ci_low": round(float(ci[0]), 4),
        "ci_high": round(float(ci[1]), 4),
        "cohen_d": round(float(d), 3),
        "t": round(float(t), 3),
        "p": round(float(p), 4),
    }

stat_df = pd.DataFrame([
    paired_rgb_gray(m, ft)
    for m in ["accuracy", "auc", "recall"]
    for ft in [False, True]
])

display(stat_df)
stat_df.to_csv(AUDIT_DIR / "exp1_stats_recomputed.csv", index=False)

## 5. Validation-selected threshold analysis

Threshold optimization is performed on validation predictions only, then applied once to the corresponding test set.

In [ ]:
from sklearn.metrics import precision_recall_curve

best_scheme, best_ft = exp1.groupby(["scheme","finetune"])["auc"].mean().idxmax()
ft_code = "ft" if bool(best_ft) else "fr"
print("Best mean-AUC configuration:", best_scheme, best_ft)

thr_rows = []
for split_i in range(5):
    run_id = f"exp1_{best_scheme}_{ft_code}_split{split_i}"
    with np.load(ART_DIR / run_id / "preds.npz", allow_pickle=False) as d:
        y_val, p_val = d["y_val"], d["p_val"]
        y_test, p_test = d["y_test"], d["p_test"]

    prec, rec, thr = precision_recall_curve((y_val==1).astype(int), p_val)
    f1s = 2 * prec * rec / (prec + rec + 1e-9)
    best_t = float(thr[np.nanargmax(f1s[:-1])]) if len(thr) else 0.5

    yhat05 = (p_test >= 0.5).astype(int)
    yhatopt = (p_test >= best_t).astype(int)

    thr_rows.append({
        "split": split_i,
        "threshold_val": best_t,
        "recall_0.5": recall_score(y_test, yhat05),
        "recall_opt": recall_score(y_test, yhatopt),
        "precision_opt": precision_score(y_test, yhatopt, zero_division=0),
        "fn_0.5": int(((p_test < 0.5) & (y_test==1)).sum()),
        "fn_opt": int(((p_test < best_t) & (y_test==1)).sum()),
    })

thr_df = pd.DataFrame(thr_rows)
display(thr_df)
thr_df.to_csv(AUDIT_DIR / "exp3_threshold_recomputed.csv", index=False)

fn0 = int(thr_df["fn_0.5"].sum())
fn1 = int(thr_df["fn_opt"].sum())
print("Total FN @0.5:", fn0)
print("Total FN @optimized:", fn1)
print("Overall FN reduction:", f"{100*(1-fn1/fn0):.2f}%")

## 6. Brier score and simulated-prevalence PPV

Low-prevalence values are mathematical simulations from observed TPR/FPR, not external clinical validation.

In [ ]:
def precision_at_prevalence(y, p, thr, prevalence):
    y = (np.asarray(y)==1).astype(int)
    yhat = (np.asarray(p)>=thr).astype(int)
    tpr = ((yhat==1)&(y==1)).sum() / max((y==1).sum(), 1)
    fpr = ((yhat==1)&(y==0)).sum() / max((y==0).sum(), 1)
    return (tpr*prevalence) / (tpr*prevalence + fpr*(1-prevalence) + 1e-9)

cal_rows = []
for split_i in range(5):
    run_id = f"exp1_{best_scheme}_{ft_code}_split{split_i}"
    with np.load(ART_DIR / run_id / "preds.npz", allow_pickle=False) as d:
        y, p = d["y_test"], d["p_test"]

    cal_rows.append({
        "split": split_i,
        "brier": brier_score_loss((y==1).astype(int), p),
        "prec@50%": precision_at_prevalence(y,p,0.5,0.50),
        "prec@5%": precision_at_prevalence(y,p,0.5,0.05),
        "prec@1%": precision_at_prevalence(y,p,0.5,0.01),
    })

cal_df = pd.DataFrame(cal_rows)
display(cal_df.round(6))
cal_df.to_csv(AUDIT_DIR / "exp3b_calibration_recomputed.csv", index=False)

## 7. Structural audit of all 25 CNN model archives

Each CNN run must retain:

- `best_model.keras`
- `latest.weights.h5`
- `history.csv`
- `preds.npz`

The `.keras` archive must contain readable `metadata.json`, `config.json`, and embedded `model.weights.h5`.

In [ ]:
import tempfile
import h5py

cnn_runs = original_results[original_results.tag != "baseline_color"].run_id.tolist()
assert len(cnn_runs) == 25

model_rows = []

for run_id in cnn_runs:
    run_dir = ART_DIR / run_id
    model_path = run_dir / "best_model.keras"
    checkpoint_path = run_dir / "latest.weights.h5"
    history_path = run_dir / "history.csv"
    preds_path = run_dir / "preds.npz"

    row = {
        "run_id": run_id,
        "best_model_exists": model_path.exists(),
        "latest_weights_exists": checkpoint_path.exists(),
        "history_exists": history_path.exists(),
        "preds_exists": preds_path.exists(),
    }

    if not all([model_path.exists(), checkpoint_path.exists(), history_path.exists(), preds_path.exists()]):
        row.update(status="FAIL", reason="missing expected artifact")
        model_rows.append(row)
        continue

    try:
        with zipfile.ZipFile(model_path, "r") as z:
            members = set(z.namelist())
            required = {"metadata.json", "config.json", "model.weights.h5"}
            missing = required - members

            row["keras_zip_test"] = z.testzip() is None
            row["keras_required_members"] = len(missing) == 0

            if missing:
                row.update(status="FAIL", reason=f"missing members: {sorted(missing)}")
                model_rows.append(row)
                continue

            metadata_json = json.loads(z.read("metadata.json").decode("utf-8"))
            config_json = json.loads(z.read("config.json").decode("utf-8"))
            row["keras_version_saved"] = metadata_json.get("keras_version")
            row["model_class_name"] = config_json.get("class_name")

            with tempfile.NamedTemporaryFile(suffix=".h5") as tmp:
                tmp.write(z.read("model.weights.h5"))
                tmp.flush()
                with h5py.File(tmp.name, "r") as h5:
                    row["embedded_h5_readable"] = True
                    row["h5_top_groups"] = ",".join(sorted(h5.keys()))

        with h5py.File(checkpoint_path, "r") as h5:
            row["checkpoint_h5_readable"] = True

        hist = pd.read_csv(history_path)
        row["history_rows"] = len(hist)
        row["history_nonempty"] = len(hist) > 0

        row["model_sha256"] = hashlib.sha256(model_path.read_bytes()).hexdigest()
        row["checkpoint_sha256"] = hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()

        ok = all([
            row["keras_zip_test"],
            row["keras_required_members"],
            row["embedded_h5_readable"],
            row["checkpoint_h5_readable"],
            row["history_nonempty"],
        ])
        row.update(status="PASS" if ok else "WARNING", reason="")

    except Exception as e:
        row.update(status="FAIL", reason=repr(e))

    model_rows.append(row)

model_audit = pd.DataFrame(model_rows)
display(model_audit[[
    "run_id", "status", "keras_version_saved",
    "model_class_name", "history_rows"
]])
model_audit.to_csv(AUDIT_DIR / "keras_model_structural_audit_25runs.csv", index=False)

print(model_audit.status.value_counts())
assert (model_audit.status == "PASS").all()
print("PASS: 25/25 CNN model archives are structurally readable.")

## 8. Optional full Keras deserialization

Set `RUN_FULL_MODEL_LOAD_TEST = True` to deserialize all 25 CNN models with TensorFlow/Keras.  
It is disabled by default because it is slower and more memory-intensive than structural validation.

In [ ]:
RUN_FULL_MODEL_LOAD_TEST = False

if RUN_FULL_MODEL_LOAD_TEST:
    import tensorflow as tf
    rows = []

    for run_id in cnn_runs:
        model_path = ART_DIR / run_id / "best_model.keras"
        try:
            model = tf.keras.models.load_model(model_path, compile=False)
            rows.append({
                "run_id": run_id,
                "status": "PASS",
                "params": int(model.count_params()),
                "input_shape": str(model.input_shape),
                "output_shape": str(model.output_shape),
                "reason": "",
            })
            del model
            tf.keras.backend.clear_session()
        except Exception as e:
            rows.append({"run_id": run_id, "status": "FAIL", "reason": repr(e)})

    full_load_df = pd.DataFrame(rows)
    display(full_load_df)
    full_load_df.to_csv(AUDIT_DIR / "keras_full_load_audit_25runs.csv", index=False)
    print(full_load_df.status.value_counts())
else:
    print("Skipped. Set RUN_FULL_MODEL_LOAD_TEST=True for 25/25 full deserialization.")

## 9. Reconstructed support figures

These plots are regenerated from frozen numerical results and stored separately.  
They do not overwrite the original publication figures.

In [ ]:
import matplotlib.pyplot as plt

plot_df = main_summary.copy()
plot_df["condition"] = (
    plot_df["scheme"].str.upper() + " / " +
    np.where(plot_df["finetune"], "Fine-tuned", "Frozen")
)

x = np.arange(len(plot_df))
width = 0.36

fig, ax = plt.subplots(figsize=(10,5))
ax.bar(x-width/2, plot_df["accuracy_mean"], width, label="Accuracy")
ax.bar(x+width/2, plot_df["auc_mean"], width, label="ROC-AUC")
ax.set_xticks(x)
ax.set_xticklabels(plot_df["condition"], rotation=20, ha="right")
ax.set_ylim(0.85, 1.0)
ax.set_ylabel("Mean across five splits")
ax.set_title("Main Experiment: Accuracy and ROC-AUC")
ax.legend()
fig.tight_layout()
fig.savefig(AUDIT_FIG_DIR / "reproduced_main_accuracy_auc.png", dpi=160)
plt.show()

baseline_summary = (
    original_results.groupby("tag")
    .agg(accuracy=("accuracy","mean"), auc=("auc","mean"))
    .reset_index()
)

fig, ax = plt.subplots(figsize=(8,5))
x = np.arange(len(baseline_summary))
ax.bar(x-width/2, baseline_summary["accuracy"], width, label="Accuracy")
ax.bar(x+width/2, baseline_summary["auc"], width, label="ROC-AUC")
ax.set_xticks(x)
ax.set_xticklabels(baseline_summary["tag"], rotation=20, ha="right")
ax.set_ylim(0.75, 1.0)
ax.set_title("Experiment Groups and Anti-Shortcut Baselines")
ax.legend()
fig.tight_layout()
fig.savefig(AUDIT_FIG_DIR / "reproduced_baseline_comparison.png", dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(8,5))
ax.plot(thr_df["split"], thr_df["recall_0.5"], marker="o", label="Recall @ 0.5")
ax.plot(thr_df["split"], thr_df["recall_opt"], marker="o", label="Recall @ validation-optimal threshold")
ax.set_xticks(thr_df["split"])
ax.set_xlabel("Split")
ax.set_ylabel("Recall")
ax.set_title("Validation-Selected Threshold Trade-off")
ax.legend()
fig.tight_layout()
fig.savefig(AUDIT_FIG_DIR / "reproduced_threshold_tradeoff.png", dpi=160)
plt.show()

## 10. Provenance SHA-256 manifest and final verdict

Key frozen inputs are fingerprinted so later modifications can be detected.

In [ ]:
def sha256_file(path, block_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(block_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

provenance_paths = [RESULTS_CSV, LEDGER_PATH] + split_files
prov_rows = []

for p in provenance_paths:
    if p.exists():
        prov_rows.append({
            "path": str(p),
            "size_bytes": p.stat().st_size,
            "sha256": sha256_file(p),
        })

provenance_df = pd.DataFrame(prov_rows)
display(provenance_df)
provenance_df.to_csv(AUDIT_DIR / "provenance_sha256_manifest.csv", index=False)

final_verdict = pd.DataFrame([{
    "audit_timestamp_utc": pd.Timestamp.utcnow().isoformat(),
    "prediction_runs_expected": 30,
    "prediction_runs_pass": int((prediction_audit.status=="PASS").sum()),
    "split_files_expected": 5,
    "split_files_pass": int((split_audit.status=="PASS").sum()),
    "cnn_models_expected": 25,
    "cnn_structural_pass": int((model_audit.status=="PASS").sum()),
    "max_prediction_metric_absdiff": float(prediction_audit.max_metric_absdiff.max()),
    "group_leakage_detected": bool(
        (split_audit[[
            "group_train_val_overlap",
            "group_train_test_overlap",
            "group_val_test_overlap"
        ]].to_numpy() > 0).any()
    ),
    "filepath_leakage_detected": bool(
        (split_audit[[
            "path_train_val_overlap",
            "path_train_test_overlap",
            "path_val_test_overlap"
        ]].to_numpy() > 0).any()
    ),
    "overall_status": "PASS",
}])

display(final_verdict)
final_verdict.to_csv(AUDIT_DIR / "FINAL_REPRODUCIBILITY_VERDICT.csv", index=False)

print("="*60)
print("FINAL CORE REPRODUCIBILITY VERDICT: PASS")
print("- 30/30 prediction archives reproduce reported metrics")
print("- 5/5 frozen splits pass group and filepath leakage audit")
print("- 25/25 CNN Keras archives pass structural binary audit")
print("="*60)

# Publication interpretation

If all mandatory cells finish with **PASS**, the archived evidence supports the following statement:

> The reported numerical results are reproducible from the frozen prediction archives without retraining. All 30 prediction archives are readable and reproduce the recorded test metrics within floating-point tolerance. All five frozen data partitions show zero overlap of inferred groups and exact image filepaths between training, validation, and test subsets. All 25 CNN runs retain structurally readable Keras model archives, checkpoints, training histories, and prediction outputs.

### Limitations that should remain explicit

1. The five partitions are repeated splits of one dataset, not five independent cohorts.
2. Group IDs are patient-like when available but also use `IMG_*` fallback identifiers; avoid claiming universal patient-level splitting.
3. Exact CUDA, cuDNN, and NVIDIA driver versions from the original session were not archived.
4. Simulated low-prevalence PPV is not external clinical validation.
5. Deployment latency from the original study was measured in Colab, not on a smartphone CPU.
6. External validation on an independent dataset remains separate from internal reproducibility.